# Train 'Neo' Wake-Word for J.A.R.V.I.S.

Run this notebook on **Google Colab** with a free T4 GPU:

**Runtime → Change runtime type → T4 GPU**

Total time: ~45-60 minutes. Output: `neo.onnx` (~1.2 MB).

In [ ]:
# 1. Install dependencies
!pip install -q openwakeword[train] piper-tts audiomentations datasets soundfile librosa onnx onnxruntime

In [ ]:
# 2. Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Configuration — edit if needed
TARGET_PHRASE = 'Neo'
ALT_PHRASES   = ['Hey Neo', 'Neo one']
MODEL_NAME    = 'neo'
N_POSITIVES   = 10000
N_NEGATIVES   = 30000
TRAIN_STEPS   = 50000

In [ ]:
# 4. Generate positives (synthetic TTS)
import os
os.makedirs('training_data/positive', exist_ok=True)
os.makedirs('training_data/negative', exist_ok=True)
os.makedirs('output', exist_ok=True)

from openwakeword.data import generate_synthetic_clips_with_piper
phrases = [TARGET_PHRASE] + ALT_PHRASES
generate_synthetic_clips_with_piper(
    phrases=phrases,
    output_dir='training_data/positive',
    n_samples_per_phrase=N_POSITIVES // len(phrases),
    voices=['en_US-lessac-medium', 'en_US-amy-medium', 'en_GB-alan-medium'],
    augment=True,
)

In [ ]:
# 5. Download negatives (FMA + AudioSet, ~12 GB)
import openwakeword.utils
openwakeword.utils.download_negative_datasets(
    output_dir='training_data/negative',
    n_samples=N_NEGATIVES,
)

In [ ]:
# 6. Train (~30-45 min on T4)
from openwakeword.train import train_model_from_config_file
config = {
    'model_name':            MODEL_NAME,
    'target_phrase':         TARGET_PHRASE,
    'training_data_path':    'training_data',
    'output_dir':            'output',
    'n_positive_samples':    N_POSITIVES,
    'n_negative_samples':    N_NEGATIVES,
    'n_validation_samples':  1000,
    'training_steps':        TRAIN_STEPS,
    'batch_size':            128,
    'learning_rate':         1e-4,
    'target_fp_per_hour':    0.5,
    'export_onnx':           True,
    'export_tflite':         False,
}
train_model_from_config_file(config)

In [ ]:
# 7. Download the trained model
from google.colab import files
files.download('output/neo.onnx')

## Deploy in Jarvis

Copy the downloaded `neo.onnx` into:
```
.venv\Lib\site-packages\openwakeword\resources\models\neo.onnx
```

Set in `.env` (or `config.py`):
```
OWW_MODEL_NAME=neo
```

Restart Jarvis. Say **"Neo"** to wake.